# Evaluating conformal predictions

Evaluation is independent of the estimator. The evaluators receive observed targets and predictions that have already been produced.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tinyconformal.evaluation import (
    CPSEvaluator,
    PanelEvaluator,
    RegressorEvaluator,
)

## Tabular regression

`RegressorEvaluator` consumes the two-column array returned by `predict_interval()`.

In [2]:
y_test = np.array([10.0, 12.0, 15.0, 18.0])
intervals = np.array([
    [8.0, 12.0],
    [10.0, 14.0],
    [13.0, 17.0],
    [14.0, 17.0],
])

RegressorEvaluator.evaluate(
    y_true=y_test,
    intervals=intervals,
    coverage=0.9,
)

,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,0.9,0.75,3.75,8.75,4


## Panel forecasts

`PanelEvaluator` aligns observations and forecasts using `unique_id` and `ds`. Interval pairs following the `<model>-lo-<coverage>` and `<model>-hi-<coverage>` convention are discovered automatically.

In [3]:
forecast = pd.DataFrame({
    "unique_id": ["a", "a", "b", "b"],
    "ds": pd.to_datetime(["2026-01-01", "2026-01-02"] * 2),
    "LinearRegression": [10.0, 12.0, 15.0, 16.0],
    "LinearRegression-lo-90": [8.0, 10.0, 13.0, 14.0],
    "LinearRegression-hi-90": [12.0, 14.0, 17.0, 18.0],
})

observed = pd.DataFrame({
    # Deliberately use a different order to demonstrate key-based alignment.
    "unique_id": ["b", "a", "b", "a"],
    "ds": pd.to_datetime(["2026-01-02", "2026-01-01", "2026-01-01", "2026-01-02"]),
    "y": [19.0, 10.0, 15.0, 12.0],
})

PanelEvaluator.evaluate_interval(
    y_true=observed,
    forecast=forecast,
)

,model,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,LinearRegression,0.9,0.75,4.0,9.0,4.0


## Custom interval-column names

When columns do not follow the standard convention, declare each pair explicitly:

In [4]:
custom_forecast = forecast.rename(columns={
    "LinearRegression-lo-90": "lower",
    "LinearRegression-hi-90": "upper",
})

PanelEvaluator.evaluate_interval(
    y_true=observed,
    forecast=custom_forecast,
    intervals={"LinearRegression": ("lower", "upper", 0.9)},
)

,model,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,LinearRegression,0.9,0.75,4.0,9.0,4.0


## Predictive distributions

`evaluate_interval()` assesses selected interval views, while `evaluate_distribution()` scores the complete predictive distribution with CRPS.


In [5]:
from sklearn.dummy import DummyRegressor
from tinyconformal.distribution import ContinuousCrossConformalPredictiveSystem

X_calibration = np.arange(6).reshape(-1, 1)
y_calibration = np.array([8.0, 9.0, 10.0, 11.0, 12.0, 13.0])
location = DummyRegressor(strategy="mean").fit(X_calibration, y_calibration)
dispersion = DummyRegressor(strategy="constant", constant=1.0).fit(
    X_calibration, np.ones(len(X_calibration))
)
cps = ContinuousCrossConformalPredictiveSystem(location, dispersion).fit(
    X_calibration, y_calibration, cv=2
)
distribution = cps.predict_distribution(np.array([[6], [7]]))
observed = np.array([11.0, 12.0])

display(CPSEvaluator.evaluate_interval(
    observed, distribution, coverages=[0.8, 0.9]
))
CPSEvaluator.evaluate_distribution(observed, distribution, scale=observed.mean())


,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,0.8,1.0,8.0,8.0,2.0
1,0.9,1.0,8.0,8.0,2.0


,crps,n_obs,scale,ncrps
0,1.234495,2,11.5,0.107347


## Distribution evaluation for three SKUs

`PanelEvaluator` reports CRPS per SKU and normalizes it by the sample standard deviation of that SKU in the training data. The normalization scale never uses the evaluation period.


In [8]:
from mlforecast import MLForecast
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from tinyconformal.series import ContinuousTimeSeriesConformalPredictiveSystem

rng = np.random.default_rng(42)
skus = ["SKU-A", "SKU-B", "SKU-C"]
dates = pd.date_range("2026-01-01", periods=40)
train_parts = []
for index, sku in enumerate(skus):
    level = 10.0 + 10.0 * index
    trend = np.linspace(0.0, 5.0 + index, len(dates))
    seasonality = 2.0 * np.sin(np.arange(len(dates)) * 2.0 * np.pi / 7.0)
    values = level + trend + seasonality + rng.normal(0.0, 0.8 + 0.2 * index, len(dates))
    train_parts.append(pd.DataFrame({"unique_id": sku, "ds": dates, "y": values}))

full_panel = pd.concat(train_parts, ignore_index=True)
test_dates = dates[-2:]
train_panel = full_panel[~full_panel["ds"].isin(test_dates)].reset_index(drop=True)
sku_observed = full_panel[full_panel["ds"].isin(test_dates)].reset_index(drop=True)

learner = MLForecast(
    models={"LinearRegression": LinearRegression()},
    freq="D",
    lags=[1, 7],
    date_features=["dayofweek"],
)
panel_cps = ContinuousTimeSeriesConformalPredictiveSystem(
    learner=learner,
    dispersion_learner=RandomForestRegressor(
        n_estimators=50,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
).fit(
    train_panel,
    horizon=2,
    n_windows=4,
    step_size=2,
    static_features=[],
    n_jobs=1,
)
panel_forecast = panel_cps.predict_distribution(h=2)


In [10]:
display(PanelEvaluator.evaluate_interval(
    sku_observed, panel_forecast, coverages=(0.8, 0.9)
))
PanelEvaluator.evaluate_distribution(
    y_true=sku_observed,
    forecast=panel_forecast,
    train_df=train_panel,
)

,model,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,LinearRegression,0.8,0.667,3.09,4.098,6.0
1,LinearRegression,0.9,0.667,3.09,5.106,6.0


,unique_id,crps,target_std,ncrps,n_obs
0,SKU-A,0.768897,2.200610,0.349402,2
1,SKU-B,0.666166,2.260534,0.294694,2
2,SKU-C,0.415638,2.626238,0.158264,2
